In [11]:
from pathlib import Path

# falls du im notebooks-Ordner bist:
epex_dir = Path("../data/raw/epex")

print("Existiert Ordner?", epex_dir.exists())
print("Dateien:")
for p in epex_dir.glob("*"):
    print(p)

Existiert Ordner? True
Dateien:
../data/raw/epex/epex_prices_2017.json
../data/raw/epex/epex_prices_2021.json
../data/raw/epex/epex_prices_2020.json
../data/raw/epex/epex_prices_2016.json
../data/raw/epex/epex_prices_2025.json
../data/raw/epex/epex_prices_2024.json
../data/raw/epex/epex_prices_2023.json
../data/raw/epex/epex_prices_2019.json
../data/raw/epex/epex_prices_2018.json
../data/raw/epex/epex_prices_2022.json


In [12]:
from pathlib import Path
import json

epex_dir = Path("../data/raw/epex")

for path in sorted(epex_dir.glob("*.json")):
    print("\n" + "="*80)
    print(path.name)
    
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    print("Typ:", type(data))
    
    if isinstance(data, dict):
        print("Keys:", data.keys())
        
        for key, value in data.items():
            if isinstance(value, list):
                print(f"{key}: Länge={len(value)}, erste Werte={value[:5]}")
            else:
                print(f"{key}: {type(value)} -> {value}")
    
    elif isinstance(data, list):
        print("Liste Länge:", len(data))
        print("Erstes Element:", data[:1])


epex_prices_2016.json


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [13]:
import pandas as pd
from pathlib import Path
import json

epex_dir = Path("../data/raw/epex")
frames = []

for path in sorted(epex_dir.glob("*.json")):
    txt = path.read_text(encoding="utf-8", errors="replace").strip()

    if not txt:
        print(path.name, "leer, übersprungen")
        continue

    try:
        data = json.loads(txt)
    except json.JSONDecodeError:
        print(path.name, "kein JSON, übersprungen")
        continue

    timestamps = data.get("unix_seconds")
    prices = data.get("price")

    if timestamps is None or prices is None:
        print(path.name, "Felder fehlen, übersprungen. Keys:", data.keys())
        continue

    if len(timestamps) == 0 or len(prices) == 0:
        print(path.name, "keine Werte, übersprungen")
        continue

    df_year = pd.DataFrame({
        "timestamp": pd.to_datetime(timestamps, unit="s", utc=True),
        "price_da": prices,
    })

    df_year["timestamp"] = (
        df_year["timestamp"]
        .dt.tz_convert("Europe/Berlin")
        .dt.tz_localize(None)
    )

    print(path.name, "geladen:", len(df_year), "Zeilen")
    frames.append(df_year)

df_epex = (
    pd.concat(frames, ignore_index=True)
    .drop_duplicates(subset=["timestamp"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print(df_epex.head())
print(df_epex.tail())
print("Zeitraum:", df_epex["timestamp"].min(), "bis", df_epex["timestamp"].max())
print("Zeilen:", len(df_epex))

epex_prices_2016.json kein JSON, übersprungen
epex_prices_2017.json kein JSON, übersprungen
epex_prices_2018.json geladen: 8760 Zeilen
epex_prices_2019.json geladen: 8760 Zeilen
epex_prices_2020.json geladen: 8784 Zeilen
epex_prices_2021.json kein JSON, übersprungen
epex_prices_2022.json kein JSON, übersprungen
epex_prices_2023.json kein JSON, übersprungen
epex_prices_2024.json geladen: 8784 Zeilen
epex_prices_2025.json kein JSON, übersprungen
            timestamp  price_da
0 2018-01-01 00:00:00       NaN
1 2018-01-01 01:00:00       NaN
2 2018-01-01 02:00:00       NaN
3 2018-01-01 03:00:00       NaN
4 2018-01-01 04:00:00       NaN
                timestamp  price_da
35079 2024-12-31 19:00:00     67.77
35080 2024-12-31 20:00:00     35.56
35081 2024-12-31 21:00:00     15.70
35082 2024-12-31 22:00:00      9.06
35083 2024-12-31 23:00:00      0.52
Zeitraum: 2018-01-01 00:00:00 bis 2024-12-31 23:00:00
Zeilen: 35084
